# Ch.3 — The XOR Problem

> A single linear layer can only draw a straight line. XOR is the simplest problem where no straight line works. One hidden layer with a non-linear activation fixes it — and that insight scales to every deep network ever built.

**Core dataset:** Synthetic XOR (4 points)  
**Housing connection:** `Latitude` × `MedInc` interaction — coastal-high-income vs inland-low-income vs the mixed cases a linear model can't separate

## The Core Idea

A linear model partitions space with a single hyperplane:
$$\mathbf{W}^\top \mathbf{x} + b = 0$$

XOR places the two positive examples at diagonally opposite corners of the unit square. No hyperplane can separate them from the two negatives — they are **linearly inseparable**.

The fix: add a **hidden layer** that transforms the input into a new space where the classes are separable, then have the output layer draw a line in that new space.

## Running Example

For the real estate platform, districts that are *both* coastal **and** high-income are clearly premium. Districts that are *both* inland **and** low-income are clearly not. But coastal-low-income and inland-high-income sit in a confusing middle — this is XOR in disguise. We'll prove the linear model fails, then show why the hidden layer fixes it.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

sns.set_theme(style='whitegrid', palette='muted')
SEED = 42
print('Libraries loaded.')

## The Math — XOR Truth Table and Why a Line Fails

In [ ]:
# ── The canonical XOR dataset ─────────────────────────────────────────────
X_xor = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

labels = {0: 'Class 0 (●)', 1: 'Class 1 (○)'}
print('XOR truth table:')
print('  x1  x2  y')
for (x1, x2), y in zip(X_xor, y_xor):
    print(f'  {int(x1)}   {int(x2)}   {y}')

In [ ]:
# ── Plot: XOR points and attempted linear boundary ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: just the points
colors = ['#e74c3c' if y == 0 else '#2980b9' for y in y_xor]
markers = ['o' if y == 0 else 's' for y in y_xor]
for (x1, x2), c, m, y in zip(X_xor, colors, markers, y_xor):
    axes[0].scatter(x1, x2, c=c, marker=m, s=300, zorder=5,
                   label=f'Class {y}' if y not in [yv for (xv,yv) in zip(X_xor, y_xor) if (xv==X_xor[(list(y_xor)).index(y)]).all()] else '_')
axes[0].scatter(*X_xor[y_xor==0].T, c='#e74c3c', marker='o', s=300, label='Class 0', zorder=5)
axes[0].scatter(*X_xor[y_xor==1].T, c='#2980b9', marker='s', s=300, label='Class 1', zorder=5)
axes[0].set(title='XOR — Try Drawing a Separating Line!',
            xlabel='x₁', ylabel='x₂', xlim=(-0.3,1.3), ylim=(-0.3,1.3))
axes[0].legend()

# Right: logistic regression trained — it'll give up and draw something useless
lr = LogisticRegression(random_state=SEED)
lr.fit(X_xor, y_xor)

xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 200), np.linspace(-0.3, 1.3, 200))
Z_lr = lr.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1].contourf(xx, yy, Z_lr, alpha=0.2, cmap='RdBu')
axes[1].scatter(*X_xor[y_xor==0].T, c='#e74c3c', marker='o', s=300, label='Class 0', zorder=5)
axes[1].scatter(*X_xor[y_xor==1].T, c='#2980b9', marker='s', s=300, label='Class 1', zorder=5)
axes[1].set(title=f'Logistic Regression — Accuracy: {accuracy_score(y_xor, lr.predict(X_xor)):.0%}',
            xlabel='x₁', ylabel='x₂', xlim=(-0.3,1.3), ylim=(-0.3,1.3))
axes[1].legend()
plt.suptitle('XOR is Linearly Inseparable', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()
print(f'Logistic regression accuracy on XOR: {accuracy_score(y_xor, lr.predict(X_xor)):.0%} — no better than chance.')

## Step by Step — What the Hidden Layer Does

A hidden layer with 2 neurons and ReLU activation transforms the 2D input into a 2D hidden space where XOR becomes linearly separable.

**Architecture:**
```
Input (2D) → Dense(2, ReLU) → Dense(1, Sigmoid) → ŷ
```

Each hidden neuron draws one linear boundary. Two neurons = two boundaries = enough to carve out the XOR-positive region.

In [ ]:
# ── Manual two-layer network — hand-chosen weights for intuition ───────────
# These weights are NOT trained — they are chosen by inspection to show
# how a 2-neuron hidden layer can represent XOR.

def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def two_layer_forward(X, W1, b1, W2, b2):
    """Forward pass: Input -> ReLU hidden layer -> Sigmoid output."""
    h     = relu(X @ W1 + b1)          # shape (n, 2)
    y_hat = sigmoid(h @ W2 + b2)       # shape (n, 1)
    return h, y_hat.ravel()

# Hand-crafted weights that solve XOR
W1 = np.array([[1., 1.],
               [1., 1.]])
b1 = np.array([[-0.5, -1.5]])
W2 = np.array([[1.], [-2.]])
b2 = np.array([[-0.5]])

h, y_hat = two_layer_forward(X_xor, W1, b1, W2, b2)

print('Input → Hidden representation → Output')
print(f'{"x1,x2":>8}  {"h1,h2":>12}  {"ŷ":>6}  {"y":>4}')
for x, hi, p, yi in zip(X_xor, h, y_hat, y_xor):
    print(f'{str(x.astype(int)):>8}  {str(hi.round(1)):>12}  {p:.3f}  {yi:>4}')

In [ ]:
# ── Visualise the hidden space transformation ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot original XOR space
axes[0].scatter(*X_xor[y_xor==0].T, c='#e74c3c', marker='o', s=300, label='Class 0', zorder=5)
axes[0].scatter(*X_xor[y_xor==1].T, c='#2980b9', marker='s', s=300, label='Class 1', zorder=5)
axes[0].set(title='Original Input Space\n(not linearly separable)',
            xlabel='x₁', ylabel='x₂', xlim=(-0.3,1.3), ylim=(-0.3,1.3))
axes[0].legend()

# Plot hidden space — the 4 points in (h1, h2) coordinates
axes[1].scatter(*h[y_xor==0].T, c='#e74c3c', marker='o', s=300, label='Class 0', zorder=5)
axes[1].scatter(*h[y_xor==1].T, c='#2980b9', marker='s', s=300, label='Class 1', zorder=5)
# Draw a separating line in hidden space
hx = np.linspace(-0.1, 1.1, 100)
axes[1].plot(hx, 0.5 - hx * 0.5, color='#27ae60', linewidth=2,
             linestyle='--', label='Linear boundary (now works!)')
for (hi_pt, yi) in zip(h, y_xor):
    axes[1].annotate(f'({hi_pt[0]:.1f},{hi_pt[1]:.1f})', hi_pt,
                    textcoords='offset points', xytext=(5, 5), fontsize=9)
axes[1].set(title='Hidden Layer Space\n(linearly separable!)',
            xlabel='h₁', ylabel='h₂')
axes[1].legend()

plt.suptitle('How the Hidden Layer Transforms the Feature Space', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()
print('The hidden layer mapped the 4 XOR points into a space where a straight line separates them.')

In [ ]:
# ── sklearn MLPClassifier solves XOR with training ────────────────────────
mlp = MLPClassifier(
    hidden_layer_sizes=(4,),    # 4 hidden neurons — more than needed, but converges faster
    activation='relu',
    solver='adam',
    max_iter=10000,
    random_state=SEED
)
mlp.fit(X_xor, y_xor)
print(f'MLP accuracy on XOR: {accuracy_score(y_xor, mlp.predict(X_xor)):.0%}')

# Plot learned decision boundary
fig, ax = plt.subplots(figsize=(6, 5))
Z_mlp = mlp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z_mlp, alpha=0.25, cmap='RdBu')
ax.scatter(*X_xor[y_xor==0].T, c='#e74c3c', marker='o', s=300, label='Class 0', zorder=5)
ax.scatter(*X_xor[y_xor==1].T, c='#2980b9', marker='s', s=300, label='Class 1', zorder=5)
ax.set(title='MLP Decision Boundary — Curved!\n(logistic regression gets a line)',
       xlabel='x₁', ylabel='x₂', xlim=(-0.3,1.3), ylim=(-0.3,1.3))
ax.legend()
plt.tight_layout()
plt.show()

## The Hyperparameter Dial — Hidden Units

Each hidden neuron adds one "crease" to the decision boundary. XOR needs 2 creases. For a real problem with 20,640 housing districts and 8 features, you need many more.

In [ ]:
# ── Hidden unit sweep on XOR ───────────────────────────────────────────────
unit_counts = [1, 2, 3, 4, 8]
fig, axes = plt.subplots(1, len(unit_counts), figsize=(16, 3.5))

for ax, n_units in zip(axes, unit_counts):
    m = MLPClassifier(hidden_layer_sizes=(n_units,), activation='relu',
                     max_iter=10000, random_state=SEED)
    m.fit(X_xor, y_xor)
    acc = accuracy_score(y_xor, m.predict(X_xor))

    Z = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
    ax.scatter(*X_xor[y_xor==0].T, c='#e74c3c', marker='o', s=200, zorder=5)
    ax.scatter(*X_xor[y_xor==1].T, c='#2980b9', marker='s', s=200, zorder=5)
    ax.set(title=f'{n_units} unit(s)\nAcc: {acc:.0%}',
           xlim=(-0.3,1.3), ylim=(-0.3,1.3), xticks=[], yticks=[])

plt.suptitle('Decision Boundary vs Number of Hidden Units', y=1.05, fontsize=12)
plt.tight_layout()
plt.show()
print('With 1 unit: still linear, still fails.  With 2+: can solve XOR.')

## Housing Connection — XOR in Real Data

In the California Housing data, we can construct a proxy XOR interaction: districts are interesting (labelled 1) if they are **high-income AND coastal** OR **low-income AND inland**, but boring (0) in the mixed cases. A logistic regression on two features fails; an MLP succeeds.

In [ ]:
# ── XOR-like interaction in housing data ──────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

# Binarise: coastal = Latitude < median (more southern = more coastal in CA)
# and high_income = MedInc > median
lat_med    = df['Latitude'].median()
inc_med    = df['MedInc'].median()
df['coastal']     = (df['Latitude'] < lat_med).astype(int)
df['high_income'] = (df['MedInc']   > inc_med).astype(int)

# XOR label: interesting = (coastal AND high_income) OR (inland AND low_income)
df['xor_label'] = ((df['coastal'] == df['high_income'])).astype(int)

print('XOR label distribution:')
print(df['xor_label'].value_counts())
print(f'\nBase rate: {df["xor_label"].mean():.1%} — near 50/50 so accuracy is meaningful')

X_h = df[['coastal', 'high_income']].values.astype(float)
y_h = df['xor_label'].values

Xtr, Xte, ytr, yte = train_test_split(X_h, y_h, test_size=0.2, random_state=SEED)

In [ ]:
# ── Compare logistic regression vs MLP on the housing XOR task ────────────
lr_h = LogisticRegression(random_state=SEED)
lr_h.fit(Xtr, ytr)

mlp_h = MLPClassifier(hidden_layer_sizes=(8,), activation='relu',
                      max_iter=5000, random_state=SEED)
mlp_h.fit(Xtr, ytr)

print('Housing XOR task — test accuracy')
print(f'  Logistic Regression: {accuracy_score(yte, lr_h.predict(Xte)):.1%}')
print(f'  MLP (8 hidden units): {accuracy_score(yte, mlp_h.predict(Xte)):.1%}')
print('\nLogistic regression is near chance. The MLP captures the interaction.')

## What Can Go Wrong — Trap Demo

### Trap: Linear activation throughout collapses to a single linear layer

If you use a linear (identity) activation for the hidden layer, then stacking layers does nothing — the composition of linear functions is still linear. No amount of depth helps.

In [ ]:
# ── Trap: linear activation in hidden layer ────────────────────────────────
# MLPClassifier's 'identity' activation is the linear activation.
mlp_linear = MLPClassifier(
    hidden_layer_sizes=(16, 16, 16),  # 3 hidden layers — but all linear
    activation='identity',
    max_iter=10000,
    random_state=SEED
)
mlp_linear.fit(X_xor, y_xor)
acc_linear = accuracy_score(y_xor, mlp_linear.predict(X_xor))

mlp_relu = MLPClassifier(
    hidden_layer_sizes=(4,),
    activation='relu',
    max_iter=10000,
    random_state=SEED
)
mlp_relu.fit(X_xor, y_xor)
acc_relu = accuracy_score(y_xor, mlp_relu.predict(X_xor))

print('XOR accuracy comparison:')
print(f'  3 linear hidden layers (16,16,16): {acc_linear:.0%}  ← still fails!')
print(f'  1 ReLU hidden layer (4 units):     {acc_relu:.0%}  ← works')
print('\nThe non-linearity is the key ingredient — depth alone does nothing without it.')

In [ ]:
# ── Trap: zero initialisation (symmetry problem) ───────────────────────────
# Show manually that zero-init makes all hidden neurons identical

W1_zero = np.zeros((2, 4))   # all weights zero
b1_zero = np.zeros((1, 4))
W2_zero = np.zeros((4, 1))
b2_zero = np.zeros((1, 1))

h_zero, out_zero = two_layer_forward(X_xor, W1_zero, b1_zero, W2_zero, b2_zero)
print('\nHidden layer outputs with ZERO initialisation:')
print('Each row is one XOR input; each column is one hidden neuron.')
print(h_zero)
print('\nAll hidden neurons produce identical outputs — they will learn identical features.')
print('This is the symmetry problem. Random init breaks it.')

## Exercises

**Exercise 1 — Minimum hidden units for XOR**  
Train `MLPClassifier(hidden_layer_sizes=(n,))` for `n ∈ [1, 2, 3, 4]` on the XOR dataset, 50 random seeds each. What is the minimum `n` that achieves 100% accuracy reliably (>= 80% of seeds)? Why does `n=2` sometimes fail?

**Exercise 2 — Activation function comparison**  
Try `activation ∈ ['relu', 'tanh', 'logistic']` with `hidden_layer_sizes=(4,)` on XOR. Plot the decision boundary for each. Which converges fastest? Why does `logistic` (sigmoid) sometimes struggle compared to `tanh`?

**Exercise 3 — Real housing XOR: add more features**  
In the housing XOR task above, we used only 2 binary features. Now use all 8 original features with the same `xor_label` target. Compare `LogisticRegression` vs `MLPClassifier(hidden_layer_sizes=(32, 16))`. Does giving logistic regression more features help it solve an inherently non-linear problem?

In [ ]:
# ── Exercise 1 scaffold — minimum units for reliable XOR ──────────────────
from sklearn.utils import check_random_state

results = {}   # n_units -> list of accuracies across seeds
for n_units in [1, 2, 3, 4]:
    accs = []
    for seed in range(50):
        m = MLPClassifier(hidden_layer_sizes=(n_units,), activation='relu',
                         max_iter=10000, random_state=seed)
        m.fit(X_xor, y_xor)
        accs.append(accuracy_score(y_xor, m.predict(X_xor)))
    results[n_units] = accs

# YOUR CODE: print the fraction of seeds achieving 100% for each n_units
# ...

In [ ]:
# ── Exercise 2 scaffold — activation function comparison ───────────────────
activations = ['relu', 'tanh', 'logistic']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, act in zip(axes, activations):
    m = MLPClassifier(hidden_layer_sizes=(4,), activation=act,
                     max_iter=10000, random_state=SEED)
    m.fit(X_xor, y_xor)
    acc = accuracy_score(y_xor, m.predict(X_xor))

    # YOUR CODE: plot the decision boundary contour for each activation
    ax.set(title=f'{act}  (acc: {acc:.0%})', xlabel='x₁', ylabel='x₂')

plt.suptitle('Activation Function vs Decision Boundary', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ── Exercise 3 scaffold — housing XOR with all 8 features ─────────────────
X_all8 = df[housing.feature_names].values
y_xor_h = df['xor_label'].values

Xtr8, Xte8, ytr8, yte8 = train_test_split(X_all8, y_xor_h, test_size=0.2, random_state=SEED)
sc = StandardScaler()
Xtr8 = sc.fit_transform(Xtr8)
Xte8 = sc.transform(Xte8)

# YOUR CODE:
# 1. Fit LogisticRegression — does more features help at all?
# 2. Fit MLPClassifier(hidden_layer_sizes=(32, 16))
# 3. Compare test accuracy
# Expected: LR is still near 50-55%; MLP gets close to 100%
# ...